# Lab 2: Systolic arrays

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rht0111/edge-ai-arch-lab/blob/students/labs/lab2_systolic.ipynb)
&nbsp; Background: [course page for Lab 2](https://rht0111.github.io/edge-ai-arch-lab/labs/lab2/) &nbsp; Time: 30 min

Peak TOPS assumes every MAC is busy every cycle. This lab measures how far
from that a systolic array really is on Transformer shapes, and how much the
choice of dataflow matters.

**Goals**

1. Watch an output-stationary (OS) array compute a small GEMM, cycle by cycle.
2. Measure utilization on awkward shapes such as 197 tokens.
3. Derive a cycle model for weight-stationary (WS) from the simulator and compare the two dataflows.

In [ ]:
# Setup: installs archsim in JupyterLite or Colab; does nothing on a local install.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["archsim", "ipywidgets"])
elif "google.colab" in sys.modules:
    %pip install -q "archsim @ git+https://github.com/rht0111/edge-ai-arch-lab"

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt

from archsim import systolic, viz, workloads
from archsim.checks import check

## 1. Output-stationary, step by step

Each PE (i, j) owns one output C[i, j]. Row i of A enters from the left,
delayed by i cycles; column j of B enters from the top, delayed by j cycles.
At every cycle each PE multiplies the two values passing through it, adds the
product to its accumulator and forwards A to the right and B downwards.

Press play, or drag the slider. Orange PEs perform a MAC in that cycle.

In [ ]:
A = np.arange(1, 13).reshape(4, 3)       # 4 x 3
B = np.arange(1, 13).reshape(3, 4) % 5   # 3 x 4
res = systolic.simulate_os_tile(A, B, rows=4, cols=4, trace=True)
print("correct:", np.array_equal(res.out, A @ B), "| cycles:", res.cycles,
      "| utilization:", round(res.utilization(4, 4), 2))
viz.array_widget(res, "os")

In [ ]:
viz.busy_plot(res).set_title("fill, steady state, drain");

With K = 3 the array is never full. Check that a tile of m x n outputs with
reduction length K takes `K + m + n - 2` cycles (`systolic.os_tile_cycles`),
and convince yourself why from the skew.

## 2. Awkward shapes

A whole GEMM is a sequence of tiles. `systolic.gemm(M, K, N, rows, cols, "os")`
returns its cycles and utilization (tiles do not overlap in this model).

In [ ]:
def show(shapes, rows, cols, dataflow="os"):
    for name, (M, K, N) in shapes.items():
        g = systolic.gemm(M, K, N, rows, cols, dataflow)
        print(f"{name:<22}{rows}x{cols} {dataflow}: {g['cycles']:>9} cycles, utilization {g['utilization']:.2f}")

shapes = {"scores 197x64x197": (197, 64, 197), "qkv 197x192x576": (197, 192, 576),
          "fc2 197x768x192": (197, 768, 192), "aligned 192x64x192": (192, 64, 192)}
show(shapes, 4, 4)
show(shapes, 16, 16)

In [ ]:
Ms = np.arange(1, 130)
fig, ax = plt.subplots(figsize=(7, 3))
for size in (8, 16, 32):
    ax.plot(Ms, [systolic.gemm(int(M), 64, 192, size, size)["utilization"] for M in Ms], label=f"{size}x{size}")
ax.axvline(197 - 128, color="grey", ls=":")
ax.set_xlabel("M (tokens)"); ax.set_ylabel("utilization"); ax.set_title("K=64, N=192, OS"); ax.legend();

Two effects show up: the sawtooth (a partly filled last tile) and the
fill/drain overhead (K is short compared with rows + cols). Which one hurts
the 32x32 array more at M = 197?

## 3. Weight-stationary

In WS each PE holds one weight W[k, j], loaded before compute, one array row
per cycle. Rows of A stream in from the left (skewed), partial sums flow down
and leave at the bottom. A tile covers up to rows x cols *weights* and streams
all M rows of A.

In [ ]:
Wt = np.arange(1, 13).reshape(3, 4) % 5 + 1     # 3 x 4 stationary weights
X = np.arange(18).reshape(6, 3) % 4               # 6 rows of A, streamed
ws = systolic.simulate_ws_tile(X, Wt, rows=4, cols=4, trace=True)
print("correct:", np.array_equal(ws.out, X @ Wt), "| cycles:", ws.cycles, "| of which loading:", ws.load_cycles)
viz.array_widget(ws, "ws")

**Task 1.** Run `systolic.simulate_ws_tile` on a few shapes, look at
`.cycles`, and find a closed form for the cycles of a WS tile that streams
`m` rows of A through `k` x `n` stationary weights (k <= rows, n <= cols).

In [ ]:
for m, k, n in [(1, 1, 1), (4, 1, 1), (1, 4, 1), (1, 1, 4), (6, 3, 4), (10, 4, 2)]:
    r = systolic.simulate_ws_tile(np.ones((m, k)), np.ones((k, n)), 4, 4)
    print(f"m={m:<3} k={k} n={n}  ->  {r.cycles} cycles")

In [ ]:
def ws_tile_cycles(m, k, n):
    # TODO: closed form, including the weight-loading cycles
    raise NotImplementedError("TODO")

In [ ]:
check("lab2.ws_tile", ws_tile_cycles)

**Task 2.** Cycles of a full (M x K) @ (K x N) GEMM in WS on a rows x cols
array: tile K by rows and N by cols, every tile streams all M rows. Partial
sums of different K tiles are added in the output buffer for free.

In [ ]:
def ws_gemm_cycles(M, K, N, rows, cols):
    # TODO: loop over the tiles and add up ws_tile_cycles
    raise NotImplementedError("TODO")

In [ ]:
check("lab2.ws_gemm", ws_gemm_cycles)

## 4. OS or WS?

Compare the two dataflows on every GEMM of a ViT-Tiny block.

In [ ]:
block = [op for op in workloads.vit_block() if op.is_gemm]
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, size in zip(axes, (4, 16)):
    names = [op.name for op in block]
    os_u = [systolic.gemm(*op.shape, size, size, "os")["utilization"] for op in block]
    ws_u = [op.macs / op.count / (ws_gemm_cycles(*op.shape, size, size) * size * size) for op in block]
    x = np.arange(len(block))
    ax.bar(x - 0.2, os_u, 0.4, label="output-stationary")
    ax.bar(x + 0.2, ws_u, 0.4, label="weight-stationary")
    ax.set_xticks(x, names); ax.set_title(f"{size}x{size} array"); ax.set_ylim(0, 1)
axes[0].set_ylabel("utilization"); axes[0].legend(loc="lower left")
fig.tight_layout()

## Stretch

1. **Overlapped tiles.** Real arrays start the next tile while the previous one
   drains. Model OS with overlap as `tiles * K + (m + n - 2)` for the last tile
   only. How much of the gap to 100% closes?
2. **Shape of the array.** Keep 256 PEs but use 8 x 32 or 32 x 8. Which
   orientation suits the attention GEMMs at 197 tokens, for each dataflow?
3. What does WS buy beyond cycles? Count how often each weight is read from
   the buffer in WS and in OS for the `fc1` GEMM.

## Reflection

"Dataflow and shape mismatch decide real efficiency, not peak TOPS." Give
the utilization of the `scores` GEMM on a 16 x 16 array for both dataflows,
and say what would have to change in the workload or the array to reach 90%.